In [13]:
import time, json, httpx
import os

ENDPOINT_URL = "http://localhost:8000/query"

query = "How you doing?"
config = {
    "query" : query,
    "thread_id" : "1241"
}

with httpx.Client(timeout=60.0) as client:
    resp = client.post(ENDPOINT_URL, json=config)
    resp.raise_for_status()
    data = resp.json()
    print(data)

{'question': 'How you doing?', 'answer': "I'm doing great, thanks for asking. I'm a large language model, so I don't have feelings or emotions like humans do, but I'm always happy to help and assist with any questions or topics you'd like to discuss. It was a nice break from our previous conversation about Kubernetes Jobs, but if you have any more questions or need further clarification on `completions` and `parallelism`, feel free to ask!", 'thought_process': ['Intent: Conversational', 'Retrieval: Skipped'], 'status': 'Response generated.', 'sources': []}


In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("../.."))

In [10]:
from app.agents.nodes.planner import planner_node
state = {
        "messages": [
            {
                "role": "user",
                "content": "What is the difference between completions and parallelism in a Kubernetes Job spec?"
            }
        ],
        "current_query": "",
        "documents": [],
        "plan": [],
        "status": "",
        "final_answer": ""
    }

result = planner_node(state)

print(result)

[Planner] User message : 'What is the difference between completions and parallelism in a Kubernetes Job spec?'
[Planner] LLM raw      : 'TECHNICAL'
[Planner] Decision     : TECHNICAL
{'current_query': 'What is the difference between completions and parallelism in a Kubernetes Job spec?', 'status': 'Technical query detected. Retrieving docs for: What is the difference between completions and parallelism in a Kubernetes Job spec?', 'plan': ['Intent: Technical', 'Retrieval: Required']}


In [ ]:
from langchain_groq import ChatGroq
from app.config import settings

llm = ChatGroq(
    model=settings.GROQ_MODEL,
    api_key=settings.GROQ_API_KEY,
    temperature=0,
)

prompt = """
You are an intent classifier.

Return ONLY one word:

CONVERSATIONAL
or
TECHNICAL

Question:
What is the difference between completions and parallelism in a Kubernetes Job spec?
"""

response = llm.invoke(prompt)

print("Model:", settings.GROQ_MODEL)
print("Response:", repr(response.content))

In [14]:
tests = [
    # Conversational
    "Hi",
    "Hello",
    "Thank you",
    "What is my name?",
    "Repeat your last answer.",
    "Can you explain that again?",

    # Technical
    "What is the difference between completions and parallelism in a Kubernetes Job spec?",
    "Explain Kubernetes Deployments.",
    "What is Intel TDX?",
    "What is BGP routing?",
    "How does kube-scheduler work?",
    "What is a StatefulSet?",

    # General (should normally be conversational)
    "Tell me a joke.",
    "Who is Virat Kohli?",
    "What is the capital of France?"
]

for q in tests:
    state = {
        "messages": [
            {
                "role": "user",
                "content": q
            }
        ],
        "current_query": "",
        "documents": [],
        "plan": [],
        "status": "",
        "final_answer": ""
    }

    result = planner_node(state)

    print("=" * 100)
    print("QUESTION :", q)
    print("DECISION :", result["current_query"])

[Planner] User message : 'Hi'
[Planner] LLM raw      : 'CONVERSATIONAL'
[Planner] Decision     : CONVERSATIONAL
QUESTION : Hi
DECISION : CONVERSATIONAL
[Planner] User message : 'Hello'
[Planner] LLM raw      : 'CONVERSATIONAL'
[Planner] Decision     : CONVERSATIONAL
QUESTION : Hello
DECISION : CONVERSATIONAL
[Planner] User message : 'Thank you'
[Planner] LLM raw      : 'CONVERSATIONAL'
[Planner] Decision     : CONVERSATIONAL
QUESTION : Thank you
DECISION : CONVERSATIONAL
[Planner] User message : 'What is my name?'
[Planner] LLM raw      : 'CONVERSATIONAL'
[Planner] Decision     : CONVERSATIONAL
QUESTION : What is my name?
DECISION : CONVERSATIONAL
[Planner] User message : 'Repeat your last answer.'
[Planner] LLM raw      : 'CONVERSATIONAL'
[Planner] Decision     : CONVERSATIONAL
QUESTION : Repeat your last answer.
DECISION : CONVERSATIONAL
[Planner] User message : 'Can you explain that again?'
[Planner] LLM raw      : 'TECHNICAL'
[Planner] Decision     : TECHNICAL
QUESTION : Can you ex